# 13 — 低周波特徴抽出: A群(広窓) / B群(周波数領域) / C群(低ランク射影)

**仮説**: SNV+SG1(7,3)+sign0.9 で CV=12.96% だが、LBは未知樹種への汎化で決まる。  
低周波成分のみで組むと、CVが悪化してもLBが改善する可能性がある（nb11-12でCV↑LB↓の逆相関確認済み）。  
**主指標**: RMSE_le170。旧ベスト=14.20%(LB19.87)、新ベスト=12.96%を参照値に使う。

In [ ]:
import sys, subprocess
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import GroupKFold

# pywt チェック → なければインストール
try:
    import pywt
    print(f'pywt {pywt.__version__} available')
    HAS_PYWT = True
except ImportError:
    print('pywt not found, installing...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'PyWavelets',
                    '--quiet'], check=True)
    import pywt
    HAS_PYWT = True
    print(f'pywt {pywt.__version__} installed')

from src.utils import load_data, parse_spectra, get_groups, make_submission, setup_japanese_font
from src.preprocessing import snv, savitzky_golay, msc as _msc_raw

plt.rcParams['figure.dpi'] = 110
setup_japanese_font()
SEED = 42

train_df, test_df = load_data()
train_meta, y_s, X_raw, wn = parse_spectra(train_df)
test_meta, _, X_test_raw, _ = parse_spectra(test_df)
y = y_s.values.astype(float)
groups = get_groups(train_meta)
SPLITS = list(GroupKFold(n_splits=5).split(X_raw, y, groups))

# ---- Metrics ----
def rmse_all(yt, yp): return float(np.sqrt(np.mean((yt - yp)**2)))
def rmse_le(yt, yp, T=170.0):
    m = yt <= T
    return float(np.sqrt(np.mean((yt[m]-yp[m])**2))) if m.sum()>0 else np.nan

# ---- Sign selector (from nb12) ----
def corr_vec(X, yv):
    yc = yv - yv.mean(); Xc = X - X.mean(axis=0)
    num = (Xc * yc[:, None]).sum(0)
    den = np.sqrt((Xc**2).sum(0) * (yc**2).sum())
    with np.errstate(invalid='ignore', divide='ignore'):
        return np.where(den > 0, num/den, 0.0)

def make_sign_selector(sign_thresh, std_thresh):
    def selector(Xtr, ytr, gtr):
        r_list = [corr_vec(Xtr[gtr==sp], ytr[gtr==sp])
                  for sp in sorted(set(gtr)) if (gtr==sp).sum()>=3]
        r_mat = np.array(r_list)
        sc = np.abs(np.sign(r_mat).sum(0)) / len(r_list)
        return (sc >= sign_thresh) & (r_mat.std(0) <= std_thresh)
    return selector

SEL_BEST = make_sign_selector(0.9, 0.10)   # nb12 best

# ---- Band mask ----
def band_mask(center, hw=75):
    return (wn >= center - hw) & (wn <= center + hw)

# ---- References ----
REF_OLD  = 14.20  # SNV+SG1(11,2)+sign0.8  LB=19.87 (nb11)
REF_NEW  = 12.96  # SNV+SG1(7,3)+sign0.9/std0.10    (nb12)

print(f'Train: {X_raw.shape}  wn: {wn.min():.0f}-{wn.max():.0f} cm^-1')
print(f'References: old={REF_OLD}%, new={REF_NEW}%')

In [ ]:
# ==== 前処理ファクトリー ====
def make_preproc_factory(scale='snv', win=11, poly=2, deriv=1):
    assert win % 2 == 1 and win > poly
    def factory(Xtr):
        ref = Xtr.mean(axis=0) if scale == 'msc' else None
        def preproc(X):
            X = X.astype(float)
            if   scale == 'none': Xs = X
            elif scale == 'snv':  Xs = snv(X)
            elif scale == 'msc':  Xs = _msc_raw(X, reference=ref)
            return Xs if deriv == 0 else savitzky_golay(Xs, win, poly, deriv)
        return preproc
    return factory

def make_gaussian_factory(sigma, deriv=0, sg_win=11, sg_poly=2):
    def factory(Xtr):
        def preproc(X):
            Xs = snv(X.astype(float))
            Xs = gaussian_filter1d(Xs, sigma=sigma, axis=1)
            return Xs if deriv == 0 else savitzky_golay(Xs, sg_win, sg_poly, deriv)
        return preproc
    return factory

def make_fft_factory(base_factory, cutoff_frac):
    def factory(Xtr):
        pp = base_factory(Xtr)
        def preproc(X):
            Xpp = pp(X)
            n = Xpp.shape[1]
            n_keep = max(1, int(n * cutoff_frac))
            X_f = np.fft.rfft(Xpp, axis=1)
            X_f[:, n_keep:] = 0
            return np.fft.irfft(X_f, n=n, axis=1)
        return preproc
    return factory

def make_wavelet_factory(base_factory, level, wavelet='db4'):
    def factory(Xtr):
        pp = base_factory(Xtr)
        def preproc(X):
            Xpp = pp(X)
            out = [pywt.wavedec(row, wavelet, level=level)[0] for row in Xpp]
            return np.array(out)
        return preproc
    return factory

def make_downsample_factory(base_factory, factor):
    def factory(Xtr):
        pp = base_factory(Xtr)
        def preproc(X):
            return pp(X)[:, ::factor]
        return preproc
    return factory

# ==== 次元削減 (fold-internal) ====
def make_pca_dr(n_components):
    def dr_fn(Xtr, ytr):
        pca = PCA(n_components=n_components, random_state=SEED)
        pca.fit(Xtr)
        return pca.transform
    return dr_fn

def make_pls_dr(n_components):
    def dr_fn(Xtr, ytr):
        pls = PLSRegression(n_components=n_components)
        pls.fit(Xtr, ytr.reshape(-1, 1))
        return lambda X: pls.transform(X)
    return dr_fn

# ==== CV runner (factory + optional dim_red_fn) ====
ET_KW = dict(n_estimators=300, max_features=0.3, random_state=SEED, n_jobs=-1)

def run_cv3(preproc_factory, model_fn=None, feat_fn=None, dim_red_fn=None):
    if model_fn is None:
        model_fn = lambda: ExtraTreesRegressor(**ET_KW)
    fold_rows, oof_y_list, oof_p_list = [], [], []
    for fi, (tr, va) in enumerate(SPLITS):
        pp = preproc_factory(X_raw[tr])
        Xtr = pp(X_raw[tr]); Xva = pp(X_raw[va])
        ytr, yva = y[tr], y[va]; gtr = groups[tr]
        n_feat = Xtr.shape[1]
        if feat_fn is not None:
            sel = feat_fn(Xtr, ytr, gtr)
            Xtr = Xtr[:, sel]; Xva = Xva[:, sel]
            n_feat = int(sel.sum())
        if dim_red_fn is not None:
            tf = dim_red_fn(Xtr, ytr)
            Xtr = tf(Xtr); Xva = tf(Xva)
            n_feat = Xtr.shape[1]
        m = model_fn(); m.fit(Xtr, ytr)
        pred = m.predict(Xva)
        fold_rows.append({'fold': fi+1, 'n_feat': n_feat,
                          'RMSE_all':   rmse_all(yva, pred),
                          'RMSE_le170': rmse_le(yva,  pred)})
        oof_y_list.append(yva); oof_p_list.append(pred)
    oof_y = np.concatenate(oof_y_list)
    oof_p = np.concatenate(oof_p_list)
    return fold_rows, oof_y, oof_p

def make_row(label, fold_rows, oof_y, oof_p):
    df = pd.DataFrame(fold_rows)
    m = df[['RMSE_all','RMSE_le170']].mean()
    return {'label': label, 'n_feat': fold_rows[0]['n_feat'],
            'RMSE_le170': round(m['RMSE_le170'],2),
            'RMSE_all':   round(m['RMSE_all'],  2),
            'folds': [round(r['RMSE_le170'],2) for r in fold_rows]}

def run_and_print(label, pf, ffn=None, drfn=None):
    rows, oy, op = run_cv3(pf, feat_fn=ffn, dim_red_fn=drfn)
    r = make_row(label, rows, oy, op)
    vs_old = r['RMSE_le170'] - REF_OLD
    vs_new = r['RMSE_le170'] - REF_NEW
    print(f'  {label:45s} le170={r["RMSE_le170"]:5.2f}%  '
          f'all={r["RMSE_all"]:5.2f}%  vs_old={vs_old:+.2f}  vs_new={vs_new:+.2f}')
    return r, oy, op

all_results = []  # 全群の結果を蓄積
all_oofs    = {}  # label -> (oof_y, oof_p)
print('Factories and run_cv3 defined.')

## A群: 広窓スムージング系

SNV+SG1 のウィンドウを広くするほど低周波寄りになる。  
さらに Gaussian平滑化 と ダウンサンプリング（次元削減）を試す。

In [ ]:
print('=== Group A: Wide Window + Gaussian + Downsampling ===')

# --- A1: 広窓 SG1 ---
A1_configs = [
    ('SNV+SG1(21,2)',  make_preproc_factory('snv', 21, 2, 1)),
    ('SNV+SG1(31,2)',  make_preproc_factory('snv', 31, 2, 1)),
    ('SNV+SG1(41,2)',  make_preproc_factory('snv', 41, 2, 1)),
    ('SNV+SG1(51,2)',  make_preproc_factory('snv', 51, 2, 1)),
    ('SNV+SG1(41,3)',  make_preproc_factory('snv', 41, 3, 1)),
]

# --- A2: Gaussian 平滑化 ---
A2_configs = [
    ('SNV+Gauss(s5)',       make_gaussian_factory(5,  deriv=0)),
    ('SNV+Gauss(s10)',      make_gaussian_factory(10, deriv=0)),
    ('SNV+Gauss(s20)',      make_gaussian_factory(20, deriv=0)),
    ('SNV+Gauss(s5)+SG1',  make_gaussian_factory(5,  deriv=1)),
    ('SNV+Gauss(s10)+SG1', make_gaussian_factory(10, deriv=1)),
]

# --- A3: 広窓 + ダウンサンプリング ---
base41 = make_preproc_factory('snv', 41, 2, 1)
A3_configs = [
    ('SNV+SG1(41,2)+skip4',  make_downsample_factory(base41, 4)),
    ('SNV+SG1(41,2)+skip8',  make_downsample_factory(base41, 8)),
]

# --- A4: 広窓 + 一貫性選択 (nb12 best selection) ---
A4_configs = [
    ('SNV+SG1(31,2)+sign0.9', make_preproc_factory('snv', 31, 2, 1)),
    ('SNV+SG1(41,2)+sign0.9', make_preproc_factory('snv', 41, 2, 1)),
]

all_A = A1_configs + A2_configs + A3_configs
# A4 には feat_fn=SEL_BEST を別途渡す

for lbl, pf in all_A:
    r, oy, op = run_and_print(lbl, pf)
    all_results.append(r); all_oofs[lbl] = (oy, op)

# A4: 広窓 + 選択
for lbl, pf in A4_configs:
    r, oy, op = run_and_print(lbl, pf, ffn=SEL_BEST)
    all_results.append(r); all_oofs[lbl] = (oy, op)

a_results = [r for r in all_results if r['label'].startswith('SNV+SG1(') or
             r['label'].startswith('SNV+Gauss')]
best_a = min(a_results, key=lambda r: r['RMSE_le170'])
print(f'\n  A群 最良: {best_a["label"]} = {best_a["RMSE_le170"]:.2f}%')

## B群: 周波数領域

### B1: FFT 低域通過フィルタ
SNV→FFT→低周波係数のみ保持→IFFT（元の波数次元に戻す）。  
カットオフ率が小さいほど低周波寄り（スペクトルが滑らか）。

### B2: ウェーブレット近似成分
SNV→DWT→approximation係数のみ（詳細成分を捨てる）。  
分解レベルが深いほど低周波寄り（次元も小さくなる）。

In [ ]:
print('=== Group B1: FFT low-pass ===')

snv_factory = make_preproc_factory('snv', 11, 2, 0)   # SNV only (no deriv)

B1_configs = [
    ('SNV+FFT(5%)',   make_fft_factory(snv_factory, 0.05)),
    ('SNV+FFT(10%)',  make_fft_factory(snv_factory, 0.10)),
    ('SNV+FFT(20%)',  make_fft_factory(snv_factory, 0.20)),
    ('SNV+FFT(40%)',  make_fft_factory(snv_factory, 0.40)),
]

for lbl, pf in B1_configs:
    r, oy, op = run_and_print(lbl, pf)
    all_results.append(r); all_oofs[lbl] = (oy, op)

# FFT + 一貫性選択 (best 1-2)
B1_with_sel = [
    ('SNV+FFT(10%)+sign0.9', make_fft_factory(snv_factory, 0.10)),
    ('SNV+FFT(20%)+sign0.9', make_fft_factory(snv_factory, 0.20)),
]
for lbl, pf in B1_with_sel:
    r, oy, op = run_and_print(lbl, pf, ffn=SEL_BEST)
    all_results.append(r); all_oofs[lbl] = (oy, op)

print('\n=== Group B2: Wavelet approximation (db4) ===')

B2_configs = [
    ('SNV+Wavelet(db4,L2)', make_wavelet_factory(snv_factory, 2, 'db4')),
    ('SNV+Wavelet(db4,L3)', make_wavelet_factory(snv_factory, 3, 'db4')),
    ('SNV+Wavelet(db4,L4)', make_wavelet_factory(snv_factory, 4, 'db4')),
    ('SNV+Wavelet(db4,L5)', make_wavelet_factory(snv_factory, 5, 'db4')),
    ('SNV+Wavelet(sym5,L3)', make_wavelet_factory(snv_factory, 3, 'sym5')),
    ('SNV+Wavelet(sym5,L4)', make_wavelet_factory(snv_factory, 4, 'sym5')),
]

for lbl, pf in B2_configs:
    r, oy, op = run_and_print(lbl, pf)
    all_results.append(r); all_oofs[lbl] = (oy, op)

b_results = [r for r in all_results if r['label'].startswith('SNV+FFT') or
             r['label'].startswith('SNV+Wavelet')]
best_b = min(b_results, key=lambda r: r['RMSE_le170'])
print(f'\n  B群 最良: {best_b["label"]} = {best_b["RMSE_le170"]:.2f}%')

## C群: 低ランク射影

### C1: PCA (fold-internal, 教師なし)
SNV+SG1(7,3) 後に PCA。上位主成分 ≈ なだらかな大局変動。

### C2: PLS (fold-internal, 教師あり)
SNV+SG1(7,3) 後に PLS。含水率と共変動する成分を優先的に抽出。

In [ ]:
print('=== Group C1: PCA (fold-internal) ===')

# nb12 best preprocessing: SNV+SG1(7,3)
pf_best = make_preproc_factory('snv', 7, 3, 1)

C1_configs = [
    ('SNV+SG1(7,3)+PCA(3)',  make_pca_dr(3)),
    ('SNV+SG1(7,3)+PCA(5)',  make_pca_dr(5)),
    ('SNV+SG1(7,3)+PCA(10)', make_pca_dr(10)),
    ('SNV+SG1(7,3)+PCA(20)', make_pca_dr(20)),
    ('SNV+SG1(7,3)+PCA(50)', make_pca_dr(50)),
]

for lbl, drfn in C1_configs:
    r, oy, op = run_and_print(lbl, pf_best, drfn=drfn)
    all_results.append(r); all_oofs[lbl] = (oy, op)

print('\n=== Group C2: PLS (fold-internal, supervised) ===')

C2_configs = [
    ('SNV+SG1(7,3)+PLS(2)',  make_pls_dr(2)),
    ('SNV+SG1(7,3)+PLS(3)',  make_pls_dr(3)),
    ('SNV+SG1(7,3)+PLS(5)',  make_pls_dr(5)),
    ('SNV+SG1(7,3)+PLS(7)',  make_pls_dr(7)),
    ('SNV+SG1(7,3)+PLS(10)', make_pls_dr(10)),
]

for lbl, drfn in C2_configs:
    r, oy, op = run_and_print(lbl, pf_best, drfn=drfn)
    all_results.append(r); all_oofs[lbl] = (oy, op)

# PCA/PLS の variance 可視化（fold1 の学習側で確認）
pp_vis = pf_best(X_raw)  # 全訓練データで fit（可視化用のみ）
X_pp_vis = pp_vis(X_raw)
pca_vis = PCA(n_components=10).fit(X_pp_vis)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# 寄与率
axes[0].bar(range(1,11), pca_vis.explained_variance_ratio_*100, color='steelblue')
axes[0].set_xlabel('PCA Component'); axes[0].set_ylabel('Explained variance (%)')
axes[0].set_title('SNV+SG1(7,3) PCA 寄与率 (Top 10)')
axes[0].grid(True, alpha=0.3, axis='y')

# 上位3成分のプロファイル
for i in range(3):
    axes[1].plot(wn, pca_vis.components_[i],
                 label=f'PC{i+1} ({pca_vis.explained_variance_ratio_[i]*100:.1f}%)',
                 alpha=0.8)
axes[1].axvline(4760, color='red', lw=1, ls='--', label='4760 cm^-1')
axes[1].set_xlabel('wavenumber (cm^-1)')
axes[1].set_ylabel('loading')
axes[1].set_title('PCA 上位3成分のプロファイル\n(低周波的=なだらか か?)')
axes[1].legend(fontsize=8); axes[1].invert_xaxis()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/c1_pca_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

c_results = [r for r in all_results if '+PCA(' in r['label'] or '+PLS(' in r['label']]
best_c = min(c_results, key=lambda r: r['RMSE_le170'])
print(f'\n  C群 最良: {best_c["label"]} = {best_c["RMSE_le170"]:.2f}%')

## D群: 低周波 × 汎化核 4760帯 の併用

B/C群の最良低周波特徴に、水の物理的吸収帯(4760 cm⁻¹)の帯平均値を追加。

In [ ]:
print('=== Group D: Low-freq + 4760 band hybrid ===')

def make_hybrid_factory(lowfreq_pf, band_center=4760, hw=75):
    """低周波特徴 + 4760帯帯平均 を結合するファクトリー。"""
    def factory(Xtr):
        pp_lf  = lowfreq_pf(Xtr)
        snv_pp = make_preproc_factory('snv', 11, 2, 0)(Xtr)  # SNV for band
        bm = band_mask(band_center, hw)
        def preproc(X):
            Xlf   = pp_lf(X)
            Xsnv  = snv_pp(X)
            band  = Xsnv[:, bm].mean(axis=1, keepdims=True)
            return np.concatenate([Xlf, band], axis=1)
        return preproc
    return factory

# B群最良と C群最良で D群を試す
best_b_fft = [r for r in all_results if r['label'].startswith('SNV+FFT')]
best_b_fft.sort(key=lambda r: r['RMSE_le170'])
best_b_lbl = best_b_fft[0]['label'] if best_b_fft else 'SNV+FFT(10%)'

best_c_pca = [r for r in all_results if '+PCA(' in r['label']]
best_c_pca.sort(key=lambda r: r['RMSE_le170'])
best_c_lbl = best_c_pca[0]['label'] if best_c_pca else 'SNV+SG1(7,3)+PCA(10)'

# D configs
print(f'  Base B: {best_b_lbl}')
print(f'  Base C: {best_c_lbl}')

# FFT-based factory for best B FFT
fft_cutoff_best = 0.10  # default, will use actual best
for frac_str, frac in [('5%',0.05),('10%',0.10),('20%',0.20),('40%',0.40)]:
    if fft_cutoff_best == 0.10 and f'FFT({frac_str})' in best_b_lbl:
        fft_cutoff_best = frac
        break

pf_best_b = make_fft_factory(snv_factory, fft_cutoff_best)

# PCA components from best C
n_pca_best = 10  # default
for k in [3,5,10,20,50]:
    if f'PCA({k})' in best_c_lbl:
        n_pca_best = k
        break

D_configs = [
    (f'D: FFT({fft_cutoff_best:.0%})+4760', make_hybrid_factory(pf_best_b)),
    (f'D: PCA({n_pca_best})+4760', make_hybrid_factory(
        lambda Xtr, k=n_pca_best: (
            lambda pf=pf_best, dr=make_pca_dr(k): (
                lambda X: dr(pf(Xtr), None)(pf(X))
            )()
        )(),
    )),
]

# D1: FFT + 4760
r, oy, op = run_and_print(D_configs[0][0], D_configs[0][1])
all_results.append(r); all_oofs[D_configs[0][0]] = (oy, op)

# D2: PCA + 4760 (inline implementation)
d2_lbl = f'D: PCA({n_pca_best})+4760'
def make_pca_hybrid(n_comp, band_center=4760, hw=75):
    def factory(Xtr):
        pp  = pf_best(Xtr)
        Xtr_pp = pp(Xtr)
        pca = PCA(n_components=n_comp, random_state=SEED).fit(Xtr_pp)
        snv_pp = make_preproc_factory('snv', 11, 2, 0)(Xtr)
        bm = band_mask(band_center, hw)
        def preproc(X):
            Xpca  = pca.transform(pp(X))
            band  = snv_pp(X)[:, bm].mean(axis=1, keepdims=True)
            return np.concatenate([Xpca, band], axis=1)
        return preproc
    return factory

r2, oy2, op2 = run_and_print(d2_lbl, make_pca_hybrid(n_pca_best))
all_results.append(r2); all_oofs[d2_lbl] = (oy2, op2)

d_results = [r for r in all_results if r['label'].startswith('D:')]
best_d = min(d_results, key=lambda r: r['RMSE_le170'])
print(f'\n  D群 最良: {best_d["label"]} = {best_d["RMSE_le170"]:.2f}%')

## 全群 横断比較

In [ ]:
# 参照行を先頭に追加
REF_ROWS = [
    {'label': '--- 旧ベスト (nb11, LB=19.87) ---',
     'n_feat': 374, 'RMSE_le170': REF_OLD, 'RMSE_all': 17.03, 'folds': []},
    {'label': '--- 新ベスト (nb12) ---',
     'n_feat': 475, 'RMSE_le170': REF_NEW, 'RMSE_all': 16.13, 'folds': []},
]

all_with_ref = REF_ROWS + sorted(all_results, key=lambda r: r['RMSE_le170'])

print('=== 全群 横断比較 (RMSE_le170 昇順) ===')
print(f'{"Rank":>4} {"Group":>2} {"Label":>47} {"n_feat":>7} '
      f'{"RMSE_le170":>11} {"RMSE_all":>10} {"vs_old":>8} {"vs_new":>8}')
print('-' * 105)

rank = 0
for r in all_with_ref:
    if r['label'].startswith('---'):
        print(f'     {r["label"]}')
        continue
    rank += 1
    if   r['label'].startswith('SNV+SG1(') and 'skip' not in r['label']: grp = 'A'
    elif r['label'].startswith('SNV+Gauss'):                              grp = 'A'
    elif 'skip' in r['label']:                                            grp = 'A'
    elif r['label'].startswith('SNV+FFT'):                                grp = 'B'
    elif r['label'].startswith('SNV+Wavelet'):                            grp = 'B'
    elif '+PCA(' in r['label'] or '+PLS(' in r['label']:                  grp = 'C'
    elif r['label'].startswith('D:'):                                     grp = 'D'
    else:                                                                  grp = '?'
    vs_old = r['RMSE_le170'] - REF_OLD
    vs_new = r['RMSE_le170'] - REF_NEW
    mark = ' [>new]' if vs_new < -0.1 else (' [>old]' if vs_old < -0.1 else '')
    print(f'{rank:>4} {grp:>2} {r["label"]:>47} {r["n_feat"]:>7} '
          f'{r["RMSE_le170"]:>11.2f} {r["RMSE_all"]:>10.2f} '
          f'{vs_old:>+8.2f} {vs_new:>+8.2f}{mark}')

# 群別最良
print('\n=== 群別最良 ===')
for grp_name, prefix_check in [
        ('A群 (広窓)', lambda r: (r['label'].startswith('SNV+SG1(') and
                                   any(str(w) in r['label'] for w in [21,31,41,51]))
                                 or r['label'].startswith('SNV+Gauss')),
        ('B群 (FFT/Wavelet)', lambda r: r['label'].startswith('SNV+FFT') or
                                         r['label'].startswith('SNV+Wavelet')),
        ('C群 (PCA/PLS)', lambda r: '+PCA(' in r['label'] or '+PLS(' in r['label']),
        ('D群 (Hybrid)', lambda r: r['label'].startswith('D:')),
]:
    subset = [r for r in all_results if prefix_check(r)]
    if subset:
        best = min(subset, key=lambda r: r['RMSE_le170'])
        vs_old = best['RMSE_le170'] - REF_OLD
        print(f'  {grp_name:20s}: {best["label"]:45s} '
              f'le170={best["RMSE_le170"]:.2f}% ({vs_old:+.2f})')

In [ ]:
# ---- 可視化 ----
fig, ax = plt.subplots(figsize=(14, 6))

# 群ごとに色分け
grp_colors = {'A': 'steelblue', 'B': 'darkorange', 'C': 'mediumseagreen', 'D': 'crimson'}

y_pos, y_lbls, y_cols = [], [], []
sorted_all = sorted(all_results, key=lambda r: r['RMSE_le170'])
for i, r in enumerate(sorted_all):
    if   r['label'].startswith('SNV+SG1(') and any(str(w) in r['label'] for w in [21,31,41,51]):
        col = grp_colors['A']
    elif r['label'].startswith('SNV+Gauss') or 'skip' in r['label']: col = grp_colors['A']
    elif r['label'].startswith('SNV+FFT') or r['label'].startswith('SNV+Wavelet'):
        col = grp_colors['B']
    elif '+PCA(' in r['label'] or '+PLS(' in r['label']: col = grp_colors['C']
    elif r['label'].startswith('D:'): col = grp_colors['D']
    else: col = 'gray'
    y_pos.append(r['RMSE_le170'])
    y_lbls.append(r['label'][:40])
    y_cols.append(col)

ax.barh(range(len(sorted_all)), y_pos, color=y_cols, alpha=0.8)
ax.axvline(REF_OLD, color='black', ls='--', lw=1.5, label=f'旧ベスト {REF_OLD}%')
ax.axvline(REF_NEW, color='red',   ls='--', lw=1.5, label=f'新ベスト {REF_NEW}%')
ax.set_yticks(range(len(sorted_all)))
ax.set_yticklabels(y_lbls, fontsize=7)
ax.set_xlabel('RMSE_le170 (%)')
ax.set_title('全群 RMSE_le170 比較 (A:青 B:橙 C:緑 D:赤)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('../results/s13_all_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 提出候補の選定と CSV 生成

**選定基準:**  
1. **CV最良**: RMSE_le170 が最も低い設定  
2. **汎化期待 (広窓)**: A群最良 (CVは中程度でも低周波で汎化が期待できる)  
3. **汎化期待 (低ランク)**: C群最良 (PCA/PLSによる次元削減)  

CVとLBの逆相関を念頭に、CV数値だけに頼らず多様性を担保する。

In [ ]:
def predict_test_full(preproc_factory, feat_fn=None, dim_red_fn=None, clip_T=200):
    """全訓練データで学習してテスト予測 (fold-internalの全データ版)。"""
    pp = preproc_factory(X_raw)         # 全訓練でfit
    Xtr = pp(X_raw); Xte = pp(X_test_raw)
    ytr_all = y
    if feat_fn is not None:
        sel = feat_fn(Xtr, ytr_all, groups)
        Xtr = Xtr[:, sel]; Xte = Xte[:, sel]
    if dim_red_fn is not None:
        tf = dim_red_fn(Xtr, ytr_all)
        Xtr = tf(Xtr); Xte = tf(Xte)
    m = ExtraTreesRegressor(**ET_KW)
    m.fit(Xtr, ytr_all)
    pred = np.clip(m.predict(Xte), 0, clip_T)
    return pred

# ---- 1. CV最良 ----
best_cv = min(all_results, key=lambda r: r['RMSE_le170'])
print(f'CV最良: {best_cv["label"]} RMSE_le170={best_cv["RMSE_le170"]:.2f}%')

# ---- 2. A群最良 (広窓汎化期待) ----
a_wide_results = [r for r in all_results
                  if (r['label'].startswith('SNV+SG1(')
                      and any(f'({w},' in r['label'] for w in [21,31,41,51]))
                     or r['label'].startswith('SNV+Gauss')]
best_a_wide = min(a_wide_results, key=lambda r: r['RMSE_le170']) if a_wide_results else None
if best_a_wide:
    print(f'A群最良: {best_a_wide["label"]} RMSE_le170={best_a_wide["RMSE_le170"]:.2f}%')

# ---- 3. C群最良 (低ランク) ----
c_all = [r for r in all_results if '+PCA(' in r['label'] or '+PLS(' in r['label']]
best_c_lr = min(c_all, key=lambda r: r['RMSE_le170']) if c_all else None
if best_c_lr:
    print(f'C群最良: {best_c_lr["label"]} RMSE_le170={best_c_lr["RMSE_le170"]:.2f}%')

# ---- 候補リストを決定（最大3つ、重複排除）----
candidates = []
seen_labels = set()
for r in [best_cv, best_a_wide, best_c_lr]:
    if r is not None and r['label'] not in seen_labels:
        candidates.append(r)
        seen_labels.add(r['label'])

print(f'\n=== 提出候補 (合計{len(candidates)}件) ===')

# ---- 設定を label から逆引きして予測 ----
CONFIG_MAP = {}
for lbl, pf in (A1_configs + A2_configs + A3_configs + B1_configs + B2_configs):
    CONFIG_MAP[lbl] = (pf, None, None)
for lbl, pf in (A4_configs + B1_with_sel):  # sign selector variants
    CONFIG_MAP[lbl] = (pf, SEL_BEST, None)
for lbl, drfn in C1_configs:
    CONFIG_MAP[lbl] = (pf_best, None, drfn)
for lbl, drfn in C2_configs:
    CONFIG_MAP[lbl] = (pf_best, None, drfn)

for cand in candidates:
    lbl = cand['label']
    print(f'\n[{lbl}]')
    print(f'  CV: RMSE_le170={cand["RMSE_le170"]:.2f}%  RMSE_all={cand["RMSE_all"]:.2f}%')
    print(f'  folds: {cand["folds"]}')

    # 設定を特定
    if lbl in CONFIG_MAP:
        pf_c, ffn_c, drfn_c = CONFIG_MAP[lbl]
    elif lbl.startswith('D:'):
        if f'FFT' in lbl:
            pf_c = make_hybrid_factory(pf_best_b); ffn_c = None; drfn_c = None
        else:
            pf_c = make_pca_hybrid(n_pca_best); ffn_c = None; drfn_c = None
    else:
        print(f'  WARNING: config not found, skipping'); continue

    te_pred = predict_test_full(pf_c, ffn_c, drfn_c)
    tag = lbl.lower().replace(' ','_').replace('+','p').replace('(','').replace(')','')[:30]
    sub_path = f'../submissions/s13_{tag}.csv'
    sub = make_submission(test_meta, te_pred, path=sub_path)
    print(f'  Test pred: min={te_pred.min():.1f}  max={te_pred.max():.1f}  '
          f'mean={te_pred.mean():.1f}%')
    print(f'  Saved: {sub_path}')

## 総括と所見

In [ ]:
print('=' * 72)
print('13 — 低周波特徴探索 総括')
print('=' * 72)

# 群別最良
print('\n[群別最良 RMSE_le170]')
for grp_name, filt in [
    ('A群 (広窓/Gaussian)',
     lambda r: (r['label'].startswith('SNV+SG1(') and
                any(f'({w},' in r['label'] for w in [21,31,41,51]))
             or r['label'].startswith('SNV+Gauss') or 'skip' in r['label']),
    ('B群 (FFT/Wavelet)',
     lambda r: r['label'].startswith('SNV+FFT') or r['label'].startswith('SNV+Wavelet')),
    ('C群 (PCA/PLS)',
     lambda r: '+PCA(' in r['label'] or '+PLS(' in r['label']),
]:
    subset = [r for r in all_results if filt(r)]
    if subset:
        best = min(subset, key=lambda r: r['RMSE_le170'])
        worst= max(subset, key=lambda r: r['RMSE_le170'])
        vs   = best['RMSE_le170'] - REF_OLD
        print(f'  {grp_name}: {best["label"]:40s} '
              f'le170={best["RMSE_le170"]:.2f}% ({vs:+.2f})')

print(f'\n[参照]')
print(f'  旧ベスト(LB=19.87): {REF_OLD}%')
print(f'  新ベスト(nb12):     {REF_NEW}%')

print('\n[低周波化の CV への影響（vs 新ベスト12.96%）]')
a_best_r = min([r for r in all_results
                if r['label'].startswith('SNV+SG1(') and
                   any(f'({w},' in r['label'] for w in [21,31,41,51])],
               key=lambda r: r['RMSE_le170'])
b_best_r = min([r for r in all_results
                if r['label'].startswith('SNV+FFT') or r['label'].startswith('SNV+Wavelet')],
               key=lambda r: r['RMSE_le170'])
c_best_r = min([r for r in all_results
                if '+PCA(' in r['label'] or '+PLS(' in r['label']],
               key=lambda r: r['RMSE_le170'])
print(f'  A群最良: {a_best_r["RMSE_le170"]:.2f}% (vs new: {a_best_r["RMSE_le170"]-REF_NEW:+.2f}%)')
print(f'  B群最良: {b_best_r["RMSE_le170"]:.2f}% (vs new: {b_best_r["RMSE_le170"]-REF_NEW:+.2f}%)')
print(f'  C群最良: {c_best_r["RMSE_le170"]:.2f}% (vs new: {c_best_r["RMSE_le170"]-REF_NEW:+.2f}%)')

print('\n[LB検証候補と選定理由]')
for i, cand in enumerate(candidates, 1):
    vs_old = cand['RMSE_le170'] - REF_OLD
    if '+PCA(' in cand['label'] or '+PLS(' in cand['label']:
        reason = '低ランク射影: 未知樹種への汎化が最も期待できる低次元表現'
    elif any(f'({w},' in cand['label'] for w in [21,31,41,51]) or 'Gauss' in cand['label']:
        reason = '広窓スムージング: 低周波成分のみ使用, CV↑でもLB改善の可能性'
    elif 'FFT' in cand['label'] or 'Wavelet' in cand['label']:
        reason = '周波数分離: 高周波ノイズを物理的に排除'
    else:
        reason = 'CV最良: 新ベスト更新'
    print(f'  {i}. {cand["label"]} le170={cand["RMSE_le170"]:.2f}% ({vs_old:+.2f})')
    print(f'     -> {reason}')

print('\n[重要な注意]')
print('  CVとLBが逆相関した事実を踏まえ、CV最良への過信は禁物。')
print('  LB検証で初めて低周波化の汎化効果が確認できる。')
print('  提出前に予測分布(min/max/mean)がテスト統計(mean~44%)と整合するか確認。')